In [1]:
import pandas as pd
from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.models import PositionShrinkageMean

In [2]:
data = load_dataset()
pm = data.player_matches()
matches = data.matches
players = pd.read_parquet("../data/rebuild/players.parquet")

In [3]:
def predicted_skill_column_name(stat):
    return f"skill({stat})"

def skill(training, fixtures, stat):
    model = PositionShrinkageMean(k=20, scorer=lambda x: x[stat])
    model.fit(training)

    predicted = model.predict(fixtures)

    # the baseline is the global average at the players upcoming position & role
    baseline = model.baselines(fixtures)

    return (predicted / baseline).rename(predicted_skill_column_name(stat))

In [4]:
start_date = pd.to_datetime("2026-02-05")
end_date = pd.to_datetime("2026-03-14")

training = data.select_matches(matches.loc[matches["date"] < start_date, "match_id"])
test = pm[(pm["date"].between(start_date, end_date, inclusive="both")) & (pm["competition_id"] == 0)]

In [6]:
# The m most skilled players (relative to their position) at the given stat 
# for the upcoming fixtures
def most_skilled(stat, m=5):
    ratings = skill(training, test, stat)
    
    return (pd.concat([ratings, test["player_id"]], axis=1)
        .groupby("player_id")
        .agg(predicted=(predicted_skill_column_name(stat), "mean"))
        .nlargest(m, columns="predicted")
        .join(players["name"], on="player_id")
    )

In [7]:
# Now find the most skilled players in general (the players with the highest skill in any particular stat)
selected_stats = ['conversions', 'penalty_goals', 'drop_goals', 'carries', 'clean_breaks',
       'completed_tackles', 'turnovers_conceded', 'turnovers_won',
       'total_kicks', 'passes', 'carried_meters', 'offloads',
       'defenders_beaten', 'try_assists', 'tries', 'carries_per_minute',
       'total_tackles', 'dominant_tackles', 'ruck_turnovers',
       'lineouts_won', 'tackles_per_minute']

all_skills = pd.concat([skill(training, test, stat) for stat in selected_stats] + [test["player_id"]], axis=1)
best = (all_skills
    .set_index("player_id")
    .agg(["idxmax", "max"], axis=1)
    .astype({"max": "float64"})
    .groupby("player_id")
    .agg(
        best_skill=("idxmax", "first"), # idxmin should be constant over the group
        skill_rating=("max", "max")) # for rating players against each other, keep their best rating in their best skill (different ratings can occur in different roles)
    .nlargest(10, columns="skill_rating")
    .join(players["name"], on="player_id")
)

In [8]:
best

,best_skill,skill_rating,name
player_id,,,
249986,skill(total_tackles),0.969068,Noah Tisie Nene
206390,skill(turnovers_conceded),1.050945,Pablo Dimcheff
237271,skill(try_assists),1.100383,Eddie James
111408,skill(turnovers_won),1.103625,Francois Cros
175150,skill(completed_tackles),1.146623,Robert Baloucoune
243190,skill(turnovers_conceded),1.151035,Blair Murray
208272,skill(defenders_beaten),1.154008,Niccolo Cannone
174824,skill(total_tackles),1.202259,Giosue Zilocchi
128750,skill(clean_breaks),1.205009,Andrea Zambonin


The most "skilled" players seem reasonable: Fin Smith at drop_goals, Ramos at conversions, Van der Flier at lineouts won and so on.

In [9]:
(all_skills
    .quantile(q=0.1, axis=0)
    .drop("player_id")
    .sort_values()
)

skill(drop_goals)            0.243609
skill(penalty_goals)         0.250000
skill(conversions)           0.263158
skill(lineouts_won)          0.266667
skill(ruck_turnovers)        0.350877
skill(total_kicks)           0.399446
skill(try_assists)           0.416667
skill(clean_breaks)          0.563046
skill(tries)                 0.566306
skill(offloads)              0.650232
skill(dominant_tackles)      0.661660
skill(defenders_beaten)      0.663057
skill(turnovers_won)         0.667594
skill(passes)                0.671252
skill(carried_meters)        0.734245
skill(turnovers_conceded)    0.742108
skill(completed_tackles)     0.865102
skill(carries_per_minute)    0.866423
skill(tackles_per_minute)    0.869318
skill(carries)               0.876532
skill(total_tackles)         0.884491
Name: 0.1, dtype: float64

In [10]:
most_skilled("try_assists")

,predicted,name
player_id,,
158534,0.224719,Dorian Aldegheri
134796,0.285714,Joe Heyes
123723,0.289855,Danilo Fischetti
190812,0.294118,Georges-Henri Colombe
118165,0.303030,Giacomo Nicotera
